# BAGEL-7B-MoT 评测总览 · benchmark 第 4 场景（与 t2i / edit / vlm 平行）

> 本 notebook 汇总 **BAGEL-7B-MoT 底座**在项目备份中已有的全部评测结果，并盘点 bagel 评测代码模块、记录向 `benchmark/bagel/` 的**整合结果（已执行）**。
>
> **评测横跨两条线：**
> - **生成类（对应 t2i 场景）**：DPG-Bench、Qwen-Image-Bench（QIB / QIB-CN）、CLIPScore
> - **理解类（对应 vlm 场景）**：VLMEvalKit 20 项标准基准（MMBench / MME / OCRBench / DocVQA …）
>
> **数据根**：`demiwtg/bagel/`（还原自备份；权重 `BAGEL-7B-MoT` 已部署到 `env-bagel`）。
> 运行内核：`env-bagel`（torch 2.5.1+cu124，pandas 已装）。本 notebook 只读结果文件，不加载模型。

In [1]:
# ---- 环境 · 路径 · 加载工具 ----
import json, csv, re
from pathlib import Path
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.unicode.east_asian_width', True)

# notebook 位于 demiwtg/benchmark/bagel/ ；数据在同场景 data/ 下，模型包在 bagel/Bagel/
HERE     = Path.cwd().resolve()
DEMIWTG  = HERE.parents[1] if HERE.name == 'bagel' else HERE.parent / 'demiwtg'
BAGEL    = DEMIWTG / 'bagel'
DATA     = HERE / 'data' if HERE.name == 'bagel' else DEMIWTG / 'benchmark' / 'bagel' / 'data'
GEN_IMG  = DATA / 'images'
EVAL_OUT = DATA / 'outputs'
BASE_M   = EVAL_OUT / 'base' / 'BAGEL-7B-MoT'

for p in (DEMIWTG, BAGEL, GEN_IMG, BASE_M):
    print(('OK ' if p.exists() else 'MISS ')+str(p))


def load_json(p):
    with open(p) as f:
        return json.load(f)


def read_csv_df(p, sep=',', quote='"'):
    with open(p, newline='') as f:
        rows = [r for r in csv.reader(f, delimiter=sep, quotechar=quote) if r]
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows[1:], columns=rows[0])

OK /yzp/zhaozy/yangzepeng/0905/demiwtg
OK /yzp/zhaozy/yangzepeng/0905/demiwtg/bagel
OK /yzp/zhaozy/yangzepeng/0905/demiwtg/benchmark/bagel/data/images
OK /yzp/zhaozy/yangzepeng/0905/demiwtg/benchmark/bagel/data/outputs/base/BAGEL-7B-MoT


## 一、生成类评测结果（DPG-Bench / QIB / QIB-CN）

三条生成类评测口径不同：DPG=官方 mPLUG-VQA 计分，QIB(英)=CLIPScore 规则，QIB-CN=LLM 裁判榜单（与 18 个闭源/商用模型同榜）。

In [2]:
# ---- DPG-Bench（官方 mPLUG-VQA 打分）----
log = (GEN_IMG/'base_dpg'/'dpg_score.log').read_text().splitlines()
l1, l2, total, n = {}, {}, None, None
sec = None
for ln in log:
    if ln.startswith('n_images:'):
        n = int(ln.split(':')[1])
    elif 'L1 category' in ln:
        sec = 'l1'
    elif 'L2 category' in ln:
        sec = 'l2'
    elif 'DPG-Bench score' in ln:
        total = float(ln.split(':')[1]); sec = None
    elif sec and ':' in ln:
        k, v = ln.strip().split(':', 1)
        (l1 if sec == 'l1' else l2)[k.strip()] = float(v)

print(f'DPG-Bench  = {total:.2f}   (n_images={n})')
dpg = pd.DataFrame({'L1 分项': list(l1.keys()), '得分': list(l1.values())}).set_index('L1 分项')
dpg.style.format('{:.2f}')
display(dpg.sort_values('得分', ascending=False))

DPG-Bench  = 83.63   (n_images=1065)


,得分
L1 分项,
entity,87.11
other,82.40
relation,82.07
attribute,80.80
global,79.33


In [3]:
# ---- QIB（英文 · CLIPScore 规则打分）----
qib = load_json(GEN_IMG/'base_qib'/'clipscore_results.json')
print(f'QIB (EN)   CLIPScore mean = {qib["clipscore_mean"]:.4f}   (n={qib["n_images"]})')

QIB (EN)   CLIPScore mean = 32.9582   (n=1000)


In [4]:
# ---- QIB-CN（中文 · LLM 裁判榜单，BAGEL 为底座模型）----
scores = load_json(GEN_IMG/'base_qib_cn'/'judge_input_bench_scores.json')
l1 = scores['level1']
print('QIB-CN 五项一级维度 (0-100):')
display(pd.DataFrame({'维度': list(l1.keys()), '得分': [round(v,2) for v in l1.values()]}).style.format({'得分':'{:.2f}'}))
print(f"QIB-CN Overall = {scores['total']:.2f}")

QIB-CN 五项一级维度 (0-100):


,维度,得分
0,Quality,38.31
1,Aesthetics,42.48
2,Alignment,38.80
3,Real-world Fidelity,39.38
4,Creative Generation,27.62


QIB-CN Overall = 38.18


In [5]:
# ---- QIB-CN 榜单（19 模型同榜；BAGEL-7B-MoT 是底座模型）----
lb = (GEN_IMG/'base_qib_cn'/'leaderboard.md').read_text().splitlines()
rows = [[c.strip() for c in ln.strip().strip('|').split('|')] for ln in lb if ln.strip().startswith('|')]
hdr = [h.replace('**','') for h in rows[0]]
data = [[c.replace('**','') for c in r] for r in rows[2:]]  # 跳过分隔行
lbdf = pd.DataFrame(data, columns=hdr)
lbdf = lbdf.rename(columns={lbdf.columns[1]: 'Model'})
num_cols = [c for c in lbdf.columns if c not in ('Rank','Model')]
lbdf[num_cols] = lbdf[num_cols].astype(float)
display(lbdf.style
        .background_gradient(subset=num_cols, cmap='RdYlGn')
        .highlight_max(subset=num_cols, color='lightgreen')
        .format({c:'{:.2f}' for c in num_cols}))
bagel_row = lbdf[lbdf['Model'].str.contains('BAGEL', case=False)]
print('\nBAGEL 排名:', bagel_row['Rank'].iloc[0], '/ 共', len(lbdf), '；Overall', bagel_row['Overall'].iloc[0])

,Rank,Model,Quality,Aesthetics,Alignment,RW-Fid,Creative,Overall
0,1,GPT Image 2,58.65,67.53,65.85,57.38,75.23,64.69
1,2,Nano Banana 2.0,54.77,61.08,62.40,54.28,67.05,59.82
2,3,GPT Image 1.5,55.14,60.88,61.72,53.95,66.35,59.65
3,4,Nano Banana Pro,55.67,60.26,61.25,54.07,66.23,59.45
4,5,Qwen Image 2.0 Pro,54.39,58.67,59.28,51.83,64.94,57.84
5,6,Seedream 5.0,52.55,58.40,58.90,51.92,65.29,57.22
6,7,Seedream 4.5,54.41,58.72,57.31,51.69,60.64,56.78
7,8,Seedream 4.0,54.01,58.81,56.64,51.05,58.15,56.21
8,9,FLUX 2 Max,53.64,56.85,57.35,49.35,56.50,55.33
9,10,FLUX 2 Pro,52.30,56.94,57.01,47.29,56.18,54.57



BAGEL 排名: 19 / 共 19 ；Overall 38.18


## 二、理解类评测结果（VLMEvalKit 20 项标准基准）

来源 `eval_outputs/base/BAGEL-7B-MoT/`，评测时间 2026-08-09（commit 见 status.json）。下方自动聚合每个基准的 Overall / 头条指标。

In [6]:
# ---- 聚合 *_acc.csv 的 Overall 分数 ----
def overall_pct(raw):
    try:
        v = float(raw)
    except (TypeError, ValueError):
        return None
    return v*100 if v <= 1.0 else v   # 分数型(≤1)统一转百分制

acc_rows = []
for f in sorted(BASE_M.glob('BAGEL-7B-MoT_*_acc.csv')):
    name = f.name[len('BAGEL-7B-MoT_'):-len('_acc.csv')]
    df = read_csv_df(f)
    if df.empty:
        continue
    col = 'Overall' if 'Overall' in df.columns else ('accuracy' if 'accuracy' in df.columns else df.columns[-1])
    acc_rows.append({'基准': name, 'Overall(%)': overall_pct(df[col].iloc[0])})

acc = pd.DataFrame(acc_rows).dropna().sort_values('Overall(%)', ascending=False).reset_index(drop=True)

# ---- 特殊指标：MME / OCRBench / WeMath / MME-RealWorld-CN ----
special = {}
mme = read_csv_df(BASE_M/'BAGEL-7B-MoT_MME_score.csv')
if not mme.empty:
    perc, reas = float(mme['perception'].iloc[0]), float(mme['reasoning'].iloc[0])
    special['MME (perception/reasoning/total)'] = f'{perc:.1f} / {reas:.1f} / {perc+reas:.1f}'
try:
    ocr = load_json(BASE_M/'BAGEL-7B-MoT_OCRBench_score.json')
    special['OCRBench (FinalScore/Norm)'] = f'{ocr["Final Score"]} / {ocr["Final Score Norm"]}'
except Exception as e:
    special['OCRBench'] = f'读取失败 {e}'
try:
    wm = read_csv_df(BASE_M/'BAGEL-7B-MoT_WeMath_gpt4o-mini_score.csv')
    special['WeMath (Score Strict)'] = wm['Score (Strict)'].iloc[0]
except Exception as e:
    special['WeMath'] = f'读取失败 {e}'
try:
    st = load_json(BASE_M/'status.json')
    special['MME-RealWorld-CN (Overall)'] = f'{st["datasets"]["MME-RealWorld-CN"]["metrics"]["Overall"]*100:.2f}'
except Exception:
    pass

print('=== 标准 acc 类基准 (转百分制) ===')
display(acc)
print('\n=== 特殊指标 ===')
for k, v in special.items():
    print(f'  {k:<38} {v}')

=== 标准 acc 类基准 (转百分制) ===


,基准,Overall(%)
0,AI2D_TEST_NO_MASK,94.818653
1,DocVQA_VAL,94.032420
2,CountBenchQA,93.634497
3,AI2D_TEST,89.345855
4,ChartQA_TEST,83.960000
5,MMBench_DEV_EN_V11,76.006192
6,MMBench_DEV_CN_V11,73.839009
7,RealWorldQA,71.503268
8,SEEDBench_IMG,71.121417
9,CV-Bench-2D,71.008213



=== 特殊指标 ===
  MME (perception/reasoning/total)       1666.0 / 683.9 / 2349.9
  OCRBench (FinalScore/Norm)             809 / 80.9
  WeMath (Score Strict)                  0.57%
  MME-RealWorld-CN (Overall)             46.98


## 三、结果完整性与异常（需重跑/补跑的点）

自动核对官方 20 项评测套件的落盘覆盖，并标出可疑结果。

In [7]:
# ---- 评测套件覆盖率：run_eval_suite.sh 的 20 项 vs 实际落盘 ----
SUITE = ['MMBench_DEV_EN_V11','MMBench_DEV_CN_V11','MME','MME-RealWorld','MME-RealWorld-CN',
         'SEEDBench_IMG','SEEDBench2_Plus','CV-Bench-2D','CV-Bench-3D','RealWorldQA',
         'MathVista_MINI','WeMath','MathVision_MINI','OCRBench','AI2D_TEST','AI2D_TEST_NO_MASK',
         'DocVQA_VAL','ChartQA_TEST','InfoVQA_VAL','CountBenchQA']

existing = {f.name for f in BASE_M.iterdir()}
def has_result(ds):
    pats = [f'BAGEL-7B-MoT_{ds}_acc.csv', f'BAGEL-7B-MoT_{ds}_score.csv',
            f'BAGEL-7B-MoT_{ds}_score.json', f'BAGEL-7B-MoT_{ds}_rating.json',
            f'BAGEL-7B-MoT_{ds}_gpt4o-mini_score.csv']
    return any(p in existing for p in pats)

cov = pd.DataFrame({'数据集': SUITE, '已落盘': ['✅' if has_result(d) else '❌ 缺失' for d in SUITE]})
display(cov)
done = cov['已落盘'].str.startswith('✅').sum()
print(f'覆盖: {done}/{len(SUITE)} 项已落盘')

,数据集,已落盘
0,MMBench_DEV_EN_V11,✅
1,MMBench_DEV_CN_V11,✅
2,MME,✅
3,MME-RealWorld,✅
4,MME-RealWorld-CN,✅
5,SEEDBench_IMG,✅
6,SEEDBench2_Plus,✅
7,CV-Bench-2D,✅
8,CV-Bench-3D,✅
9,RealWorldQA,✅


覆盖: 18/20 项已落盘


In [8]:
# ---- 异常点标注 ----
anomalies = []
# 1) WeMath strict 近乎 0
try:
    wm = float(read_csv_df(BASE_M/'BAGEL-7B-MoT_WeMath_gpt4o-mini_score.csv')['Score (Strict)'].iloc[0].rstrip('%'))
    if wm < 5:
        anomalies.append(('WeMath', f'Score(Strict)={wm:.2f}%，InsufficientKnowledge 占 94% → 疑似答案抽取/格式不匹配，非真实能力，建议重跑'))
except Exception:
    pass
# 2) t2i 自建基准 bagel 报告 n=0
t2i_rep = DEMIWTG/'benchmark'/'t2i'/'data'/'scores_t2i_v60_V2_gpt-5.6-sol_bagel.report.json'
if t2i_rep.exists():
    r = load_json(t2i_rep)
    if not r.get('n'):
        anomalies.append(('t2i-bagel report', f"n={r.get('n')} → BAGEL 那路未跑完 (同批 gpt-image-2 有数据)"))
# 3) lora / official_cfg 变体 metrics 是否为空
for var in ('lora', 'official_cfg'):
    vdir = EVAL_OUT/var/'BAGEL-7B-MoT'
    if vdir.exists():
        empties = 0; runs = 0
        for sj in vdir.glob('*/status.json'):
            runs += 1
            try:
                d = load_json(sj)
                if not any(v.get('metrics') for v in d.get('datasets',{}).values()):
                    empties += 1
            except Exception:
                empties += 1
        anomalies.append((f'eval_outputs/{var}', f'{runs} 次运行, 其中 {empties} 次 metrics 为空 → 微调/CFG 变体评测大多未落盘'))
# 4) 覆盖率缺口
missing = [d for d in SUITE if not has_result(d)]
if missing:
    anomalies.append(('套件缺口', '缺失: ' + ', '.join(missing) + ' → 需补跑'))

if anomalies:
    display(pd.DataFrame(anomalies, columns=['项','说明']))
else:
    print('未发现异常')

,项,说明
0,WeMath,Score(Strict)=0.57%，InsufficientKnowledge 占 94...
1,t2i-bagel report,n=0 → BAGEL 那路未跑完 (同批 gpt-image-2 有数据)
2,eval_outputs/lora,"1 次运行, 其中 1 次 metrics 为空 → 微调/CFG 变体评测大多未落盘..."
3,eval_outputs/official_cfg,"4 次运行, 其中 4 次 metrics 为空 → 微调/CFG 变体评测大多未落盘..."
4,套件缺口,"缺失: MathVista_MINI, MathVision_MINI → 需补跑"


## 四、BAGEL 评测代码模块清单（已物理整合到 benchmark/bagel/）

下表为整合**后**的模块清单（2026-09-05 已执行物理移动；`data/` 下为软链，指向大体量结果/题库/权重）。路径相对 `demiwtg/`。

In [9]:
# ---- 模块清单（整合后实际位置 + 自动核对存在性）----
MODULES = {
  'A. 生成类 · DPG/QIB (benchmark/bagel/gen/)': [
    ('benchmark/bagel/gen/eval_gen.py',       '生成驱动(多卡分片) ← gen_eval/scripts/generate_t2i.py'),
    ('benchmark/bagel/gen/run_gen_eval.sh',   'DPG/QIB 编排 (base|lora × dpg|qib|all)'),
    ('benchmark/bagel/gen/eval_score_clip.py','CLIPScore 规则打分'),
    ('benchmark/bagel/gen/eval_score_dpg.py', 'DPG 官方 mPLUG-VQA 打分(需 modelscope)'),
    ('benchmark/bagel/gen/eval_score_qib.py', 'QIB-CN LLM 裁判榜单'),
    ('benchmark/bagel/gen/run_qib_cn_pipeline.sh','QIB-CN 四阶段流水线'),
    ('benchmark/bagel/gen/prep_{dl_qib,dl_mme_rw,extract_qib_prompts}.py','数据准备'),
    ('benchmark/bagel/gen/_fairseq_stub.py',  'score_dpg 的 sibling 依赖'),
    ('benchmark/bagel/gen/qib_official/',     'QIB 官方评分库(vendored, 已移入)'),
    ('benchmark/bagel/data/dpg_bench',        'DPG 题库(物理移入, 源自 ELLA 仓)'),
    ('benchmark/bagel/data/images',           '生成结果(物理移入)'),
  ],
  'B. 理解类 · VLM 套件 (benchmark/bagel/vlm/)': [
    ('benchmark/bagel/vlm/eval_vlm.py',       '单数据集评测驱动 ← Bagel/scripts/run_bagel_eval.py'),
    ('benchmark/bagel/vlm/run_eval_suite.sh', '20 数据集编排'),
    ('benchmark/bagel/vlm/run_official_mmbench.sh','MMBench 官方口径'),
    ('benchmark/bagel/vlm/VLMEvalKit/',       '框架(已物理移入)'),
    ('benchmark/bagel/data/outputs',          '结果(物理移入)'),
  ],
  'C. 生成类官方基准 (Bagel repo 自带, 留在原位)': [
    ('bagel/Bagel/eval/gen/{geneval,gedit,imgedit}/','生成/编辑官方基准'),
    ('bagel/Bagel/eval/gen/{kris,rise,wise}/','推理/编辑类基准'),
  ],
}

def exists_glob(rel):
    p = DEMIWTG/rel
    if '{' in rel:
        head = re.split(r'[{/].*', rel)[0].strip('/')
        return (DEMIWTG/head).exists()
    return p.exists() or bool(list(DEMIWTG.glob(rel)))

for group, items in MODULES.items():
    print('\n' + group)
    for it in items:
        rel, role = it if isinstance(it, tuple) else (it, '')
        mark = '✓' if exists_glob(rel) else '✗(需核对)'
        print(f'  [{mark}] {rel}')
        if role:
            print(f'        ↳ {role}')


A. 生成类 · DPG/QIB (benchmark/bagel/gen/)
  [✓] benchmark/bagel/gen/eval_gen.py
        ↳ 生成驱动(多卡分片) ← gen_eval/scripts/generate_t2i.py
  [✓] benchmark/bagel/gen/run_gen_eval.sh
        ↳ DPG/QIB 编排 (base|lora × dpg|qib|all)
  [✓] benchmark/bagel/gen/eval_score_clip.py
        ↳ CLIPScore 规则打分
  [✓] benchmark/bagel/gen/eval_score_dpg.py
        ↳ DPG 官方 mPLUG-VQA 打分(需 modelscope)
  [✓] benchmark/bagel/gen/eval_score_qib.py
        ↳ QIB-CN LLM 裁判榜单
  [✓] benchmark/bagel/gen/run_qib_cn_pipeline.sh
        ↳ QIB-CN 四阶段流水线
  [✓] benchmark/bagel/gen/prep_{dl_qib,dl_mme_rw,extract_qib_prompts}.py
        ↳ 数据准备
  [✓] benchmark/bagel/gen/_fairseq_stub.py
        ↳ score_dpg 的 sibling 依赖
  [✓] benchmark/bagel/gen/qib_official/
        ↳ QIB 官方评分库(vendored, 已移入)
  [✓] benchmark/bagel/data/dpg_bench
        ↳ DPG 题库(物理移入, 源自 ELLA 仓)
  [✓] benchmark/bagel/data/images
        ↳ 生成结果(物理移入)

B. 理解类 · VLM 套件 (benchmark/bagel/vlm/)
  [✓] benchmark/bagel/vlm/eval_vlm.py
        ↳ 单数据集评测驱动 ← Bagel/scrip

## 五、整合结果（✅ 已于 2026-09-05 执行 · 物理移动）

三场景范式平行落地：**代码物理移动**进 `benchmark/bagel/{gen,vlm}/`（含规范重命名），**大体量数据/权重用软链**放 `data/`，vendored 仓 `qib_official`/`VLMEvalKit` 一并物理移入。移动时同步改写了 4 个脚本的内部引用（`SCRIPTS=`/`OFFICIAL=`/`REPO`/`VLMKIT`/`cd`）与重命名后的调用点。

In [10]:
# ---- 最终布局 & 源→目标映射（已执行） ----
TARGET_TREE = '''benchmark/bagel/                          # 第 4 场景
├── results_review.ipynb                  # 本 notebook
├── README.md
├── gen/                                  # 生成类（DPG/QIB）— 物理移动+重命名
│   ├── eval_gen.py                       ← gen_eval/scripts/generate_t2i.py
│   ├── eval_score_clip.py                ← clip_score_eval.py
│   ├── eval_score_dpg.py                 ← score_dpg_mplug.py
│   ├── eval_score_qib.py                 ← qib_leaderboard.py
│   ├── _fairseq_stub.py                  ← 同名(sibling 依赖)
│   ├── run_gen_eval.sh / run_qib_cn_pipeline.sh
│   ├── prep_{dl_qib,dl_mme_rw,extract_qib_prompts}.py
│   └── qib_official/                     ← gen_eval/qib_official(整仓移入)
├── vlm/                                  # 理解类（VLM 套件）— 物理移动+重命名
│   ├── eval_vlm.py                       ← Bagel/scripts/run_bagel_eval.py (REPO 改硬编码)
│   ├── run_eval_suite.sh                 ← Bagel/scripts/run_eval_suite.sh
│   ├── run_official_mmbench.sh           ← 同名(无需改引用)
│   └── VLMEvalKit/                       ← bagel/VLMEvalKit(整仓移入)
└── data/                                 # 真实数据(物理移入) + models 软链
    ├── images/                           ← bagel/gen_eval/images (977M 结果)
    ├── prompts/                          ← bagel/gen_eval/prompts (178M)
    ├── dpg_bench/                        ← ELLA/dpg_bench (题库, 源仓其余已删)
    ├── outputs/                          ← bagel/eval_outputs (37M)
    ├── {modelscope,hf_home,LMUData}/     # 运行缓存(脚本已指向, 按需生成)
    └── models -> bagel/models (→Bagel/models, BAGEL-7B-MoT 28G)'''

MAPPING = [
    ('源 (移动前)','目标 (benchmark/bagel/)','处理'),
    ('gen_eval/scripts/generate_t2i.py','gen/eval_gen.py','物理移动+重命名'),
    ('gen_eval/scripts/{clip_score_eval,score_dpg_mplug,qib_leaderboard}.py','gen/eval_score_*.py','物理移动+重命名'),
    ('gen_eval/scripts/run_*.sh + prep 三个','gen/','物理移动'),
    ('gen_eval/qib_official/','gen/qib_official/','整仓物理移动'),
    ('Bagel/scripts/run_bagel_eval.py','vlm/eval_vlm.py','物理移动+REPO硬编码'),
    ('Bagel/scripts/{run_eval_suite,run_official_mmbench}.sh','vlm/','物理移动'),
    ('bagel/VLMEvalKit/','vlm/VLMEvalKit/','整仓物理移动'),
    ('gen_eval/{images,prompts}, ELLA/dpg_bench, eval_outputs','data/','物理移入(真实目录)'),
    ('bagel/{fa_build,logs,gen_eval残余,ELLA其余}','—','删除(非必要: 旧机构建产物/日志/可再克隆)'),
]
print(TARGET_TREE)
display(pd.DataFrame(MAPPING[1:], columns=MAPPING[0]))

benchmark/bagel/                          # 第 4 场景
├── results_review.ipynb                  # 本 notebook
├── README.md
├── gen/                                  # 生成类（DPG/QIB）— 物理移动+重命名
│   ├── eval_gen.py                       ← gen_eval/scripts/generate_t2i.py
│   ├── eval_score_clip.py                ← clip_score_eval.py
│   ├── eval_score_dpg.py                 ← score_dpg_mplug.py
│   ├── eval_score_qib.py                 ← qib_leaderboard.py
│   ├── _fairseq_stub.py                  ← 同名(sibling 依赖)
│   ├── run_gen_eval.sh / run_qib_cn_pipeline.sh
│   ├── prep_{dl_qib,dl_mme_rw,extract_qib_prompts}.py
│   └── qib_official/                     ← gen_eval/qib_official(整仓移入)
├── vlm/                                  # 理解类（VLM 套件）— 物理移动+重命名
│   ├── eval_vlm.py                       ← Bagel/scripts/run_bagel_eval.py (REPO 改硬编码)
│   ├── run_eval_suite.sh                 ← Bagel/scripts/run_eval_suite.sh
│   ├── run_official_mmbench.sh           ← 同名(无需改引用)
│   └── VLMEvalKit/         

,源 (移动前),目标 (benchmark/bagel/),处理
0,gen_eval/scripts/generate_t2i.py,gen/eval_gen.py,物理移动+重命名
1,"gen_eval/scripts/{clip_score_eval,score_dpg_mp...",gen/eval_score_*.py,物理移动+重命名
2,gen_eval/scripts/run_*.sh + prep 三个,gen/,物理移动
3,gen_eval/qib_official/,gen/qib_official/,整仓物理移动
4,Bagel/scripts/run_bagel_eval.py,vlm/eval_vlm.py,物理移动+REPO硬编码
5,"Bagel/scripts/{run_eval_suite,run_official_mmb...",vlm/,物理移动
6,bagel/VLMEvalKit/,vlm/VLMEvalKit/,整仓物理移动
7,"gen_eval/{images,prompts}, ELLA/dpg_bench, eva...",data/,物理移入(真实目录)
8,"bagel/{fa_build,logs,gen_eval残余,ELLA其余}",—,删除(非必要: 旧机构建产物/日志/可再克隆)


## 六、整合前必须修的两处硬编码

备份里的 bagel 评测脚本写死了**旧机器路径**和**已删除的虚拟环境**，还原到新目录 + `env-bagel` 后需替换，否则脚本跑不起来。

In [11]:
# ---- 扫描需要替换的旧路径 / 旧 env ----
pats = {
    '/tank/demiwtg': '旧绝对路径 → 改为项目根 demiwtg/ 的绝对路径或 $DEMIWTG',
    '.venv/bin/python': '已删除的 .venv → 改为 /yzp/zhaozy/yangzepeng/0905/env-bagel/bin/python',
    'MODELSCOPE_CACHE': '/tank/... 缓存路径 → 改到项目内或 /yzp 下',
    'HF_HOME': '/tank/... 缓存路径 → 同上',
}
hits = {k: [] for k in pats}
root = DEMIWTG
scan_files = (list(root.glob('benchmark/bagel/gen/*.sh')) + list(root.glob('benchmark/bagel/gen/*.py'))
              + list(root.glob('benchmark/bagel/vlm/*.sh')) + list(root.glob('benchmark/bagel/vlm/*.py'))
              + list(root.glob('bagel/Bagel/scripts/*.sh')))
for f in scan_files:
    try:
        txt = f.read_text(errors='ignore')
    except Exception:
        continue
    for k in pats:
        if k in txt:
            hits[k].append(str(f.relative_to(root)))

for k, note in pats.items():
    files = hits[k]
    print(f'\n▶ 含 "{k}" 的文件 ({len(files)}): {note}')
    for f in files[:12]:
        print('    ', f)


▶ 含 "/tank/demiwtg" 的文件 (0): 旧绝对路径 → 改为项目根 demiwtg/ 的绝对路径或 $DEMIWTG

▶ 含 ".venv/bin/python" 的文件 (0): 已删除的 .venv → 改为 /yzp/zhaozy/yangzepeng/0905/env-bagel/bin/python

▶ 含 "MODELSCOPE_CACHE" 的文件 (3): /tank/... 缓存路径 → 改到项目内或 /yzp 下
     benchmark/bagel/gen/run_qib_cn_pipeline.sh
     benchmark/bagel/gen/run_gen_eval.sh
     benchmark/bagel/gen/eval_score_dpg.py

▶ 含 "HF_HOME" 的文件 (4): /tank/... 缓存路径 → 同上
     benchmark/bagel/gen/run_qib_cn_pipeline.sh
     benchmark/bagel/gen/run_gen_eval.sh
     benchmark/bagel/gen/prep_dl_mme_rw.py
     benchmark/bagel/vlm/run_eval_suite.sh


---

### 待办 / 下一步（供决策）
1. **补跑缺口**：`MathVista_MINI`、`MathVision_MINI`（套件 20 项缺 2）；`lora` / `official_cfg` 变体大多未落盘；t2i 自建基准的 BAGEL 路 `n=0`。
2. **重跑可疑项**：`WeMath` strict≈0，需确认是否答案抽取问题。
3. ~~执行整改~~ **✅ 整合已完成**（第五节）：代码已物理移动至 `benchmark/bagel/{gen,vlm}/`，引用已改写；运行入口见 `gen/run_gen_eval.sh`、`vlm/run_eval_suite.sh`。
4. **运行时依赖**：`clip`、`modelscope`、VLMEvalKit 相关依赖尚未装进 `env-bagel`，实跑生成/打分链路前需补装；缓存目录（`MODELSCOPE_CACHE`/`HF_HOME`/`LMUData`）需按需填充。
5. **端到端冒烟**：可跑 `LIMIT=32 GPUS="0" bash gen/run_gen_eval.sh base dpg` 验通全链路（权重加载+题库+生成）。